# 12b. Do the interventions still work at future demand? 2030 demand (x2.32)

The intervention results (H1-H7) were measured at 2011 demand. The projection (notebooks 4a-4b) showed that at grown demand most trips no longer finish within 6 hours. This notebook runs the strongest interventions at that grown demand, on the published model, each paired by seed with the no-intervention run at the same demand:

| Arm | What |
|---|---|
| baseline | no intervention |
| adaptive | queue-actuated signals (H1) |
| adaptive_pricing | signals + tolls (H3; the 2011 toll schedule, not re-derived for future congestion) |
| staggered | the same demand over a 50% longer window (H6) |
| all | signals + tolls + staggered departures |

Demand grown at 4.52%/yr (JICA/DoR 2024 forecast): x1.94 for 2026, x2.32 for 2030. Horizon 6 hours. 6 seeds.

**Criterion, fixed before any run:** an arm *solves* a demand level if its completion within 6 hours is at least the 2011 baseline's and its mean time in system is no worse than the 2011 baseline's (2011 reference run in notebook 12a).

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("data/jica_net/od_matrix.csv")
tolls = glob.glob("/kaggle/input/**/results/tolls_jica.csv", recursive=True)[0]
os.makedirs("results", exist_ok=True); shutil.copy(tolls, "results/tolls_jica.csv")
shutil.copy(glob.glob("/kaggle/input/**/scripts/future_jica.py", recursive=True)[0], "scripts/future_jica.py")
if os.path.exists("results/jica_future.partial.csv"): os.remove("results/jica_future.partial.csv")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/future_jica.py", "--level", "2030", "--first-seed", "1", "--seeds", "6", "--jobs", "3", tries=3)
shutil.copy("results/jica_future.partial.csv", "/kaggle/working/future_2030.csv")

In [ ]:
run("scripts/future_jica.py", "--summary-only")